# 🩸 Blood Glucose Forecaster - Training & Evaluation

**XGBoost 2.0+ Compatible** - Uses `QuantileDMatrix`  
**Target**: `CGM_future` (30-min ahead prediction)

## Cell 1: Setup & Update

In [ ]:
# Force update XGBoost to latest version
!pip install xgboost -U -q
!pip install polars -q

import polars as pl
import xgboost as xgb
import numpy as np
import matplotlib.pyplot as plt
import gc
from sklearn.metrics import mean_squared_error, mean_absolute_error

from google.colab import drive
drive.mount('/content/drive')

print(f"\n✅ XGBoost version: {xgb.__version__}")
print(f"✅ Polars version: {pl.__version__}")

# Verify QuantileDMatrix is available
assert hasattr(xgb, 'QuantileDMatrix'), "ERROR: QuantileDMatrix not found. Update XGBoost!"
print("✅ QuantileDMatrix available!")

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================
BASE_PATH = "/content/drive/MyDrive/DiabetesKDS/data"

TRAIN_FILE = f"{BASE_PATH}/processed/train_ready_for_glucose_prediction.parquet"
TEST_FILE = f"{BASE_PATH}/processed/test_ready_for_glucose_prediction.parquet"
MODEL_OUTPUT = f"{BASE_PATH}/processed/xgboost_glucose_model.json"

TARGET = 'CGM_future'
FEATURES = [
    'CGM', 'bolus', 'carbs', 'basal',
    'time_sin', 'time_cos', 'is_weekend',
    'cgm_delta_5min', 'cgm_delta_15min',
    'age', 'weight',
    'gender_encoded', 'insulin_type_encoded'
]

print(f"📁 Train: {TRAIN_FILE}")
print(f"📁 Test:  {TEST_FILE}")
print(f"📁 Model: {MODEL_OUTPUT}")
print(f"🎯 Target: {TARGET}")
print(f"📊 Features ({len(FEATURES)}): {FEATURES}")

## Cell 2: Training (XGBoost 2.0+ with QuantileDMatrix)

In [ ]:
print("="*60)
print("🚀 TRAINING (XGBoost 2.0+ QuantileDMatrix)")
print("="*60)

# Load train data
print("\n📂 Loading training data...")
df_train = pl.read_parquet(TRAIN_FILE)
print(f"✅ Shape: {df_train.shape} ({df_train.shape[0]:,} rows)")

# Prepare arrays
X_train = df_train.select(FEATURES).to_numpy().astype(np.float32)
y_train = df_train.select(TARGET).to_numpy().ravel().astype(np.float32)
del df_train; gc.collect()
print(f"   X: {X_train.shape}, y: {y_train.shape}")

# ============================================================
# FIX: Use QuantileDMatrix (XGBoost 2.0+)
# ============================================================
print("\n⚡ Creating QuantileDMatrix (GPU optimized)...")
dtrain = xgb.QuantileDMatrix(
    data=X_train,
    label=y_train,
    feature_names=FEATURES,
    max_bin=256
)
del X_train, y_train; gc.collect()
print("✅ QuantileDMatrix created!")

# Training parameters
params = {
    'objective': 'reg:squarederror',
    'tree_method': 'hist',
    'device': 'cuda',
    'max_depth': 6,
    'learning_rate': 0.05,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'seed': 42
}

print("\n📋 Parameters:")
for k, v in params.items():
    print(f"   {k}: {v}")

# Train
print("\n⏳ Training (1000 rounds)...")
model = xgb.train(
    params=params,
    dtrain=dtrain,
    num_boost_round=1000,
    evals=[(dtrain, 'train')],
    verbose_eval=100
)

# Save model
model.save_model(MODEL_OUTPUT)
print(f"\n💾 Model saved: {MODEL_OUTPUT}")

# Clear memory
del dtrain; gc.collect()
print("\n✅ TRAINING COMPLETE!")

## Cell 3: Evaluation

In [ ]:
print("="*60)
print("📊 EVALUATION")
print("="*60)

# Load test data
print("\n📂 Loading test data...")
df_test = pl.read_parquet(TEST_FILE)
print(f"✅ Shape: {df_test.shape} ({df_test.shape[0]:,} rows)")

X_test = df_test.select(FEATURES).to_numpy().astype(np.float32)
y_test = df_test.select(TARGET).to_numpy().ravel().astype(np.float32)
del df_test; gc.collect()

# Create DMatrix for prediction (standard DMatrix is fine for inference)
dtest = xgb.DMatrix(data=X_test, feature_names=FEATURES)
del X_test; gc.collect()

# Predict
print("\n🔮 Predicting...")
y_pred = model.predict(dtest)
del dtest; gc.collect()

# ============================================================
# METRICS
# ============================================================
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)

# Clinical accuracy
abs_errors = np.abs(y_test - y_pred)
within_15 = np.mean(abs_errors <= 15) * 100
within_20 = np.mean(abs_errors <= 20) * 100

print("\n" + "="*50)
print("🎯 RESULTS")
print("="*50)
print(f"  RMSE:               {rmse:.4f} mg/dL")
print(f"  MAE:                {mae:.4f} mg/dL")
print(f"  Clinical (±15):     {within_15:.2f}%")
print(f"  Clinical (±20):     {within_20:.2f}%")
print("="*50)

In [ ]:
# Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Time series
n = 200
axes[0,0].plot(range(n), y_test[:n], 'g-', lw=2, label='Actual')
axes[0,0].plot(range(n), y_pred[:n], 'r--', lw=2, label='Predicted')
axes[0,0].set_title(f'Actual vs Predicted (RMSE: {rmse:.2f})')
axes[0,0].legend(); axes[0,0].grid(alpha=0.3)

# 2. Scatter
sample = min(50000, len(y_test))
axes[0,1].scatter(y_test[:sample], y_pred[:sample], alpha=0.2, s=2)
axes[0,1].plot([0,400], [0,400], 'r--', lw=2)
axes[0,1].set_title('Scatter'); axes[0,1].grid(alpha=0.3)

# 3. Error histogram
errors = y_test[:sample] - y_pred[:sample]
axes[1,0].hist(errors, bins=100, color='purple', alpha=0.7)
axes[1,0].axvline(0, color='red', ls='--')
axes[1,0].set_title('Error Distribution'); axes[1,0].grid(alpha=0.3)

# 4. Feature importance
imp = model.get_score(importance_type='gain')
sorted_imp = sorted(imp.items(), key=lambda x: x[1])
axes[1,1].barh([x[0] for x in sorted_imp], [x[1] for x in sorted_imp], color='teal')
axes[1,1].set_title('Feature Importance')

plt.tight_layout()
plt.show()

print("\n🎉 COMPLETE!")